# Phase 2.5: Exploratory Data Analysis
## Tiller / SumUp POS Pro

**Project:** Data Analytics Capstone — Tiller (SumUp POS Pro)  
**Dataset:** `le-wagon-da-502302.tiller`  
**Period:** Oct 2015 – Nov 2020  
**Date:** September 2026

---

## Business Context

Tiller is a digital point-of-sale (POS) and business-management platform, now part of **SumUp** and marketed as **SumUp POS Pro**. The platform combines:

- Point-of-sale functionality
- Order and payment processing
- Business management tools
- Reporting and analytics

The dataset reflects platform activity from 2015–2020, when it operated under the Tiller brand.

### Project Goal

Explore how Tiller can use platform-generated data to provide more useful **analytical insights and decision support** to business customers.

### Central Question

*How can Tiller use the data generated through its platform to provide more useful analytical insights and decision support to its business customers?*

---

## Phase 2.5 Objective

This exploratory analysis examines how business activity behaves across **stores, time, transactions, products, payments and operational dimensions**.

**Goal:** Identify meaningful patterns to determine which areas could support deeper investigation in Phase 3.

### Structure

1. **Store-level behaviour** — How do locations differ in volume and value?
2. **Temporal behaviour** — How does activity vary over time?
3. **Product and category behaviour** — What products drive value?
4. **Transaction and customer economics** — What drives order value?
5. **Payment and operational behaviour** — How are transactions paid for?

---
## Setup

In [ ]:
import subprocess
from io import StringIO
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Tiller/SumUp Design System Colors
TILLER_BLUE = '#3388FF'      # SumUp Sky Blue (primary)
TILLER_DARK = '#003C8B'      # Deep Ocean (text/accents)
TILLER_GREEN = '#00C853'     # Success/paid
TILLER_ORANGE = '#FF9800'    # Warning/in progress
TILLER_RED = '#F44336'       # Attention/anomaly
CHARCOAL = '#333333'         # Primary text
GREY_LIGHT = '#F5F5F5'       # Background
GREY_BORDER = '#E0E0E0'      # Borders/gridlines

# Professional styling aligned with SumUp Circuit UI
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['axes.edgecolor'] = GREY_BORDER
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['grid.color'] = GREY_BORDER
plt.rcParams['grid.linestyle'] = '-'
plt.rcParams['grid.linewidth'] = 0.5
plt.rcParams['text.color'] = CHARCOAL
plt.rcParams['xtick.color'] = '#666666'
plt.rcParams['ytick.color'] = '#666666'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.family'] = 'sans-serif'

CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)

def run_query(query):
    result = subprocess.run(
        ["bq", "query", "--use_legacy_sql=false", "--format=csv", query],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Query failed: {result.stderr}")
    return pd.read_csv(StringIO(result.stdout))

def save_and_show(fig, filename):
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / filename, dpi=300, bbox_inches="tight", facecolor='white')
    plt.show()
    plt.close(fig)
    print(f"  ✓ Saved: charts/{filename}")

---
## 2.5.1 Store-level Behaviour

### Context

The dataset contains **21 business locations** with substantial heterogeneity. Store 4151 alone represents **~68% of all orders** but has a much lower average order value than most other locations.

### Key Questions

- Which locations have the highest order volume?
- How does average order value vary between locations?

### Why This Matters

Aggregate metrics can be strongly influenced by location composition. Understanding store-level differences is essential before interpreting overall platform performance.

In [ ]:
query = """
SELECT id_store, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value, ROUND(AVG(m_nb_customer), 2) AS avg_customers
FROM `le-wagon-da-502302.tiller.order_data`
GROUP BY id_store
ORDER BY orders DESC
"""
store_df = run_query(query)
store_df.head(10)

In [ ]:
# Validate key figures
total_orders = store_df['orders'].sum()
store_4151 = store_df[store_df['id_store'] == 4151].iloc[0]
other_stores = store_df[store_df['id_store'] != 4151]

print(f"Total orders: {total_orders:,}")
print(f"Store 4151: {store_4151['orders']:,} orders ({store_4151['orders']/total_orders*100:.2f}%)")
print(f"Store 4151 AOV: €{store_4151['avg_order_value']:.2f}")
print(f"Other locations avg AOV: €{other_stores['avg_order_value'].mean():.2f}")

### Chart 1: Order Volume by Location

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
orders_sorted = store_df.sort_values("orders", ascending=True)
ax.barh(orders_sorted["id_store"].astype(str), orders_sorted["orders"], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_xlabel("Number of orders", color=CHARCOAL)
ax.set_ylabel("Location", color=CHARCOAL)
ax.set_title("Order Volume by Location (21 stores, Oct 2015 – Nov 2020)", fontweight='bold', color=TILLER_DARK)
ax.xaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_1_orders_by_store.png")

### Chart 2: Average Order Value by Location

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
aov_sorted = store_df.sort_values("avg_order_value", ascending=True)
colors = [TILLER_RED if sid == 4151 else TILLER_BLUE for sid in aov_sorted['id_store']]
ax.barh(aov_sorted["id_store"].astype(str), aov_sorted["avg_order_value"], color=colors, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_xlabel("Average Order Value (€)", color=CHARCOAL)
ax.set_ylabel("Location", color=CHARCOAL)
ax.set_title("Average Order Value by Location (Location 4151 highlighted)", fontweight='bold', color=TILLER_DARK)
ax.xaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_1_aov_by_store.png")

### Chart 3: Location 4151 vs Other Locations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

ax = axes[0]
ax.bar(['Location 4151', 'Other Locations (avg)'], [store_4151['orders'], other_stores['orders'].mean()], color=[TILLER_RED, TILLER_BLUE], edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("Number of orders", color=CHARCOAL)
ax.set_title("Order Volume", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[1]
ax.bar(['Location 4151', 'Other Locations (avg)'], [store_4151['avg_order_value'], other_stores['avg_order_value'].mean()], color=[TILLER_RED, TILLER_BLUE], edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("AOV (€)", color=CHARCOAL)
ax.set_title("Average Order Value", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = axes[2]
ax.bar(['Location 4151', 'Other Locations (avg)'], [store_4151['avg_customers'], other_stores['avg_customers'].mean()], color=[TILLER_RED, TILLER_BLUE], edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("Avg Customers per Order", color=CHARCOAL)
ax.set_title("Customer Count", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.suptitle("Location 4151 vs Other Locations: Key Metrics", fontweight='bold', color=TILLER_DARK, y=1.02)
save_and_show(fig, "2_5_1_store_4151_vs_others.png")

### Key Findings

- **Location 4151 dominates volume:** 871,415 orders (68.02% of total)
- **Location 4151 has lowest AOV:** €7.03 vs €57.11 average for other locations
- **Implication:** Location composition strongly affects aggregate platform metrics

---
## 2.5.2 Temporal Behaviour

### Context

Transaction activity shows clear variation over time — yearly, weekly, and hourly. Understanding these patterns is crucial for demand forecasting and operational planning.

In [ ]:
query = """
SELECT EXTRACT(YEAR FROM date_opened) AS year, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value
FROM `le-wagon-da-502302.tiller.order_data`
WHERE date_opened IS NOT NULL
GROUP BY year
ORDER BY year
"""
temporal_yearly = run_query(query)
temporal_yearly

### Chart 4: Long-term Trends (Yearly)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(temporal_yearly['year'].astype(str), temporal_yearly['orders'], color=TILLER_BLUE, alpha=0.8)
ax1.set_xlabel("Year", color=CHARCOAL)
ax1.set_ylabel("Number of orders", color=TILLER_BLUE, fontweight='bold')
ax1.set_title("Long-term Trends: Order Volume and AOV by Year", fontweight='bold', color=TILLER_DARK)
ax1.yaxis.grid(True, alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2 = ax1.twinx()
ax2.plot(temporal_yearly['year'], temporal_yearly['avg_order_value'], color=TILLER_ORANGE, marker='o', linewidth=2.5, markersize=8)
ax2.set_ylabel("Average Order Value (€)", color=TILLER_ORANGE, fontweight='bold')
ax2.spines['top'].set_visible(False)
save_and_show(fig, "2_5_2_yearly_trend.png")

In [ ]:
query = """
SELECT EXTRACT(DAYOFWEEK FROM date_opened) AS day_num, FORMAT_DATE('%A', date_opened) AS day_name, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value
FROM `le-wagon-da-502302.tiller.order_data`
WHERE date_opened IS NOT NULL
GROUP BY day_num, day_name
ORDER BY day_num
"""
temporal_weekly = run_query(query)
temporal_weekly

### Chart 5: Weekly Patterns

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(temporal_weekly['day_name'], temporal_weekly['orders'], color=TILLER_BLUE, alpha=0.8)
ax1.set_xlabel("Day of Week", color=CHARCOAL)
ax1.set_ylabel("Number of orders", color=TILLER_BLUE, fontweight='bold')
ax1.set_title("Weekly Patterns: Order Volume and AOV by Day of Week", fontweight='bold', color=TILLER_DARK)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.grid(True, alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2 = ax1.twinx()
ax2.plot(range(7), temporal_weekly['avg_order_value'], color=TILLER_ORANGE, marker='o', linewidth=2.5)
ax2.set_ylabel("Average Order Value (€)", color=TILLER_ORANGE, fontweight='bold')
ax2.spines['top'].set_visible(False)
save_and_show(fig, "2_5_2_weekly_pattern.png")

In [ ]:
query = """
SELECT EXTRACT(HOUR FROM date_opened) AS hour, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value
FROM `le-wagon-da-502302.tiller.order_data`
WHERE date_opened IS NOT NULL
GROUP BY hour
ORDER BY hour
"""
temporal_hourly = run_query(query)
temporal_hourly

### Chart 6: Hourly Patterns

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(temporal_hourly['hour'], temporal_hourly['orders'], color=TILLER_BLUE, alpha=0.8)
ax1.set_xlabel("Hour of Day", color=CHARCOAL)
ax1.set_ylabel("Number of orders", color=TILLER_BLUE, fontweight='bold')
ax1.set_title("Hourly Patterns: Order Volume and AOV by Hour", fontweight='bold', color=TILLER_DARK)
ax1.yaxis.grid(True, alpha=0.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax2 = ax1.twinx()
ax2.plot(temporal_hourly['hour'], temporal_hourly['avg_order_value'], color=TILLER_ORANGE, linewidth=2.5)
ax2.set_ylabel("Average Order Value (€)", color=TILLER_ORANGE, fontweight='bold')
ax2.spines['top'].set_visible(False)
save_and_show(fig, "2_5_2_hourly_pattern.png")

### Key Findings

- **Long-term trend:** Order volume increased 2015→2019; AOV decreased (Location 4151 growth)
- **Weekly pattern:** Sunday highest volume (168,682 orders)
- **Hourly pattern:** Clear lunch (11-13h) and dinner (18-20h) peaks

---
## 2.5.3 Product and Category Behaviour

### Context

The order-line data contains **12,145 unique products** and **456 raw categories**. Product activity is concentrated in relatively few categories.

In [ ]:
query = """
SELECT dim_category, COUNT(*) AS order_lines, ROUND(SUM(m_total_price_inc_vat), 2) AS total_value
FROM `le-wagon-da-502302.tiller.order_line`
WHERE dim_category IS NOT NULL
GROUP BY dim_category
ORDER BY order_lines DESC
LIMIT 15
"""
category_dist = run_query(query)
category_dist

### Chart 7: Top Categories by Order Lines

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
cat_sorted = category_dist.sort_values("order_lines", ascending=True)
ax.barh(cat_sorted["dim_category"], cat_sorted["order_lines"], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_xlabel("Number of order lines", color=CHARCOAL)
ax.set_ylabel("Category", color=CHARCOAL)
ax.set_title("Top 15 Product Categories by Order Lines", fontweight='bold', color=TILLER_DARK)
ax.xaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_3_top_categories.png")

In [ ]:
query = """
SELECT dim_category, COUNT(*) AS order_lines, ROUND(SUM(m_total_price_inc_vat), 2) AS total_value
FROM `le-wagon-da-502302.tiller.order_line`
WHERE dim_category IS NOT NULL
GROUP BY dim_category
ORDER BY total_value DESC
LIMIT 15
"""
category_value = run_query(query)
category_value

### Chart 8: Top Categories by Value

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
val_sorted = category_value.sort_values("total_value", ascending=True)
ax.barh(val_sorted["dim_category"], val_sorted["total_value"], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_xlabel("Total value (€)", color=CHARCOAL)
ax.set_ylabel("Category", color=CHARCOAL)
ax.set_title("Top 15 Product Categories by Total Value", fontweight='bold', color=TILLER_DARK)
ax.xaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_3_top_categories_value.png")

### Key Findings

- **High concentration:** PRESSION represents 17.8% of total order-line value
- **Taxonomy issues:** CONSIGNE (ORDER) = 1.28M lines; multiple language variants

---
## 2.5.4 Transaction and Customer Economics

### Context

Order value varies substantially based on transaction composition.

In [ ]:
query = """
SELECT CASE
    WHEN m_nb_customer = 1 THEN '1 customer'
    WHEN m_nb_customer = 2 THEN '2 customers'
    WHEN m_nb_customer BETWEEN 3 AND 4 THEN '3-4 customers'
    ELSE '5+ customers'
END AS customer_segment, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value
FROM `le-wagon-da-502302.tiller.order_data`
WHERE m_nb_customer IS NOT NULL
GROUP BY customer_segment
ORDER BY MIN(m_nb_customer)
"""
customers_grouped = run_query(query)
customers_grouped

### Chart 9: Customers per Order

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(customers_grouped['customer_segment'], customers_grouped['orders'], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
axes[0].set_ylabel("Number of orders", color=CHARCOAL)
axes[0].set_title("Order Distribution by Customer Count", fontweight='bold', color=TILLER_DARK)
axes[0].yaxis.grid(True, alpha=0.5)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].bar(customers_grouped['customer_segment'], customers_grouped['avg_order_value'], color=TILLER_GREEN, edgecolor=GREY_BORDER, linewidth=0.5)
axes[1].set_ylabel("Average Order Value (€)", color=CHARCOAL)
axes[1].set_title("AOV by Customer Count", fontweight='bold', color=TILLER_DARK)
axes[1].yaxis.grid(True, alpha=0.5)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle("Transaction Economics: Impact of Customer Count on Order Value", fontweight='bold', color=TILLER_DARK, y=1.02)
save_and_show(fig, "2_5_4_customers_per_order.png")

In [ ]:
query = """
SELECT CASE
    WHEN items_per_order BETWEEN 1 AND 2 THEN '1-2 items'
    WHEN items_per_order BETWEEN 3 AND 5 THEN '3-5 items'
    WHEN items_per_order BETWEEN 6 AND 10 THEN '6-10 items'
    ELSE '11+ items'
END AS item_segment, COUNT(*) AS orders, ROUND(AVG(order_value), 2) AS avg_order_value
FROM (
    SELECT od.id_order, od.m_cached_price AS order_value, COUNT(ol.id_order_line) AS items_per_order
    FROM `le-wagon-da-502302.tiller.order_data` od
    JOIN `le-wagon-da-502302.tiller.order_line` ol ON od.id_order = ol.id_order
    GROUP BY od.id_order, od.m_cached_price
)
GROUP BY item_segment
ORDER BY MIN(items_per_order)
"""
items_grouped = run_query(query)
items_grouped

### Chart 10: Items per Order

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(items_grouped['item_segment'], items_grouped['orders'], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
axes[0].set_ylabel("Number of orders", color=CHARCOAL)
axes[0].set_title("Order Distribution by Item Count", fontweight='bold', color=TILLER_DARK)
axes[0].yaxis.grid(True, alpha=0.5)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

axes[1].bar(items_grouped['item_segment'], items_grouped['avg_order_value'], color=TILLER_GREEN, edgecolor=GREY_BORDER, linewidth=0.5)
axes[1].set_ylabel("Average Order Value (€)", color=CHARCOAL)
axes[1].set_title("AOV by Item Count", fontweight='bold', color=TILLER_DARK)
axes[1].yaxis.grid(True, alpha=0.5)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle("Transaction Economics: Impact of Item Count on Order Value", fontweight='bold', color=TILLER_DARK, y=1.02)
save_and_show(fig, "2_5_4_items_per_order.png")

### Key Findings

- **Strong positive association:** More customers/items → higher AOV
- **Item segments:** €6.96 (1-2 items) → €111.24 (11+ items)

---
## 2.5.5 Payment and Operational Behaviour

### Context

Payment methods, order sources, and financial anomalies provide operational context.

In [ ]:
query = """
SELECT dim_type, COUNT(*) AS payments, ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM `le-wagon-da-502302.tiller.payment_data`), 2) AS pct_of_total
FROM `le-wagon-da-502302.tiller.payment_data`
WHERE dim_type IS NOT NULL
GROUP BY dim_type
ORDER BY payments DESC
"""
payment_dist = run_query(query)
payment_dist.head(10)

### Chart 11: Payment Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(payment_dist['dim_type'], payment_dist['payments'], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("Number of payments", color=CHARCOAL)
ax.set_xlabel("Payment type", color=CHARCOAL)
ax.set_title("Payment Type Distribution", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_5_payment_types.png")

In [ ]:
query = """
SELECT dim_source, COUNT(*) AS orders, ROUND(AVG(m_cached_price), 2) AS avg_order_value
FROM `le-wagon-da-502302.tiller.order_data`
WHERE dim_source IS NOT NULL
GROUP BY dim_source
ORDER BY orders DESC
"""
order_source = run_query(query)
order_source

### Chart 12: Order Sources

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(order_source['dim_source'], order_source['orders'], color=TILLER_BLUE, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("Number of orders", color=CHARCOAL)
ax.set_xlabel("Order source", color=CHARCOAL)
ax.set_title("Order Volume by Source", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
save_and_show(fig, "2_5_5_order_sources.png")

In [ ]:
# Financial anomalies
anomalies_orders = run_query("SELECT 'Orders' AS metric, SUM(CASE WHEN m_cached_price < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct FROM `le-wagon-da-502302.tiller.order_data`")
anomalies_lines = run_query("SELECT 'Order Lines' AS metric, SUM(CASE WHEN m_total_price_inc_vat < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct FROM `le-wagon-da-502302.tiller.order_line`")
anomalies_payments = run_query("SELECT 'Payments' AS metric, SUM(CASE WHEN m_amount < 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct FROM `le-wagon-da-502302.tiller.payment_data`")

print(f"Negative order values: {anomalies_orders['pct'].iloc[0]:.2f}%")
print(f"Negative order-line values: {anomalies_lines['pct'].iloc[0]:.2f}%")
print(f"Negative payment amounts: {anomalies_payments['pct'].iloc[0]:.2f}%")

### Chart 13: Financial Anomalies

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Orders', 'Order Lines', 'Payments']
pct_negative = [anomalies_orders['pct'].iloc[0], anomalies_lines['pct'].iloc[0], anomalies_payments['pct'].iloc[0]]
ax.bar(metrics, pct_negative, color=TILLER_RED, edgecolor=GREY_BORDER, linewidth=0.5)
ax.set_ylabel("Percentage of negative values (%)", color=CHARCOAL)
ax.set_title("Financial Anomalies: Percentage of Negative Values", fontweight='bold', color=TILLER_DARK)
ax.yaxis.grid(True, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, v in enumerate(pct_negative):
    ax.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=11, color=TILLER_DARK)
save_and_show(fig, "2_5_5_financial_anomalies.png")

### Key Findings

- **Payment concentration:** CARD (51.92%) and CASH (43.84%) dominate
- **Financial anomalies:** 20%+ negative values across all tables (not necessarily errors)

---
## Phase 2.5 Summary

### Key Findings

| Dimension | Key Insight |
|-----------|-------------|
| **Store-level** | Location 4151 = 68% of orders but lowest AOV (€7.03) |
| **Temporal** | Clear yearly, weekly, hourly patterns; Sunday highest volume |
| **Product/Category** | High concentration (PRESSION 17.8%); taxonomy needs validation |
| **Transaction Economics** | More customers/items → higher AOV (€6.96 → €111.24) |
| **Payment/Operational** | CARD 52%, CASH 44%; 20%+ negative values |

### Next Steps

These findings will inform the selection of a focused analytical opportunity in **Phase 3**, where we will:

1. Review evidence from Phase 2
2. Select one opportunity with clear business value
3. Define a specific business question
4. Develop a focused analytical MVP

---

**Charts Exported:** All 13 charts saved to `charts/` at 300 DPI, using SumUp Circuit UI design system colors.